## PANNs CNN14 Embeddings + SVM for Respiratory Sound Classification

This notebook uses a pretrained **PANNs CNN14** model to extract audio embeddings, then trains a Support Vector Machine (SVM) to classify the ICBHI respiratory-sound labels. The notebook evaluates the four labels as a binary **Normal vs. Abnormal** task using sensitivity, specificity, and the harmonic score (HS).

**Workflow:** load the pretrained CNN14 checkpoint → extract fixed-length audio embeddings → map crackle/wheeze annotations to four labels → scale features → reduce dimensions with PCA → train an SVM → evaluate binary HS → save the pipeline.

**Hint:** Run cells in order and confirm that the pretrained checkpoint, metadata CSV, and audio directory are available relative to the notebook's working directory.

In [19]:
import joblib
from pathlib import Path
import importlib.util
import sys
import os
import random
from tqdm import tqdm

import numpy as np
import pandas as pd

import librosa
import torch
import torchlibrosa

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


In [9]:
#from audioset_tagging_cnn_master.pytorch.models import Cnn14
SRC_DIR = Path.cwd()
PYTORCH_DIR = SRC_DIR / "audioset_tagging_cnn_master" / "pytorch"
MODELS_PATH = PYTORCH_DIR / "models.py"
LOCAL_UTILS_PATH = PYTORCH_DIR / "pytorch_utils.py"

assert MODELS_PATH.is_file(), f"Missing file: {MODELS_PATH}"
assert LOCAL_UTILS_PATH.is_file(), f"Missing file: {LOCAL_UTILS_PATH}"

# Make the local helper available under the name expected by AudioSet.
utils_spec = importlib.util.spec_from_file_location(
    "pytorch_utils", LOCAL_UTILS_PATH
)
utils_module = importlib.util.module_from_spec(utils_spec)
sys.modules["pytorch_utils"] = utils_module


# Execute the local helper module
utils_spec.loader.exec_module(utils_module)

print("Local helper loaded:", utils_module.__file__)
print("Has forward:", hasattr(utils_module, "forward"))

# Load models.py directly, without executing the package __init__.py
models_spec = importlib.util.spec_from_file_location(
    "audioset_cnn14_models", MODELS_PATH
)
models_module = importlib.util.module_from_spec(models_spec)
sys.modules[models_spec.name] = models_module
models_spec.loader.exec_module(models_module)

Cnn14 = models_module.Cnn14
print("CNN14 imported successfully:", Cnn14)



SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Local helper loaded: D:\.JKU CS courses\Master Thesis\src\audioset_tagging_cnn_master\pytorch\pytorch_utils.py
Has forward: True
CNN14 imported successfully: <class 'audioset_cnn14_models.Cnn14'>
Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [10]:
def do_mixup(x, mixup_lambda):
    """Mixup x of even indexes (0, 2, 4, ...) with x of odd indexes 
    (1, 3, 5, ...).

    Args:
      x: (batch_size * 2, ...)
      mixup_lambda: (batch_size * 2,)

    Returns:
      out: (batch_size, ...)
    """
    out = (x[0 :: 2].transpose(0, -1) * mixup_lambda[0 :: 2] + \
        x[1 :: 2].transpose(0, -1) * mixup_lambda[1 :: 2]).transpose(0, -1)
    return out


### Load the Pretrained PANNs CNN14 Model

This cell creates the CNN14 architecture, loads the pretrained AudioSet checkpoint, and places the model in evaluation mode. The configured model uses a 16 kHz sample rate and returns a learned audio embedding.

**Hint:** needed files and directories for PANNs CNN14:
<br>
```
project/
  └── src/
       ├── 05.pt__cnn14+svm.ipynb
       ├── Cnn14_mAP=0.431.pth
       └── audioset_tagging_cnn_master/   #directory
```
<br>

1. Check that <b>`Cnn14_mAP=0.431.pth`</b> must be downloaded and placed inside <b>`src`</b> directory and accessible from the current working directory. <u>[Click here to download Cnn14_mAP=0.431.pth](https://zenodo.org/records/3987831)</u>

2. Check that <b>`audioset_tagging_cnn_master`</b> directory must be cloned or downloaded from Qiuqiangkong's GitHub, then  placed inside <b>`src`</b> directory. <u>[Click here to visit Qiuqiangkong GitHub](https://github.com/qiuqiangkong/audioset_tagging_cnn)</u>

    
The pretrained checkpoint must match the architecture configuration. `model.eval()` and `torch.no_grad()` are used to extract features without updating model weights.

In [11]:
model = Cnn14(
    sample_rate=16000,
    window_size=1024,
    hop_size=320,
    mel_bins=64,
    fmin=50,
    fmax=8000,
    classes_num=527
)

checkpoint = torch.load("Cnn14_mAP=0.431.pth", map_location=device)
model.load_state_dict(checkpoint["model"])
model.to(device)
model.eval()

Cnn14(
  (spectrogram_extractor): Spectrogram(
    (stft): STFT(
      (conv_real): Conv1d(1, 513, kernel_size=(1024,), stride=(320,), bias=False)
      (conv_imag): Conv1d(1, 513, kernel_size=(1024,), stride=(320,), bias=False)
    )
  )
  (logmel_extractor): LogmelFilterBank()
  (spec_augmenter): SpecAugmentation(
    (time_dropper): DropStripes()
    (freq_dropper): DropStripes()
  )
  (bn0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv_block1): ConvBlock(
    (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv_block2): ConvBlock(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (c

### Extract Fixed-Length PANNs Embeddings

`extract_panns_features()` loads each audio file at 16 kHz and pads or truncates it to 10 seconds before passing it through CNN14. The resulting embedding is converted to a NumPy vector.

**Hint:** Padding/truncation makes the input duration consistent, but it may remove content from longer files or add silence to shorter files. Keep this preprocessing identical for training and inference.

In [12]:


def extract_panns_features(path, max_len=10):
    y, sr = librosa.load(path, sr=16000)

    # pad / trim to fixed length
    if len(y) < sr * max_len:
        y = np.pad(y, (0, sr * max_len - len(y)))
    else:
        y = y[:sr * max_len]

    x = torch.tensor(y).float().unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(x, None)

    embedding = output["embedding"].cpu().numpy().flatten()

    return embedding

### Map Crackle and Wheeze Annotations to Class Labels

The function maps the two binary annotations to four classes:

- `0`: Normal (neither crackle nor wheeze)
- `1`: Crackle only
- `2`: Wheeze only
- `3`: Both crackle and wheeze

**Hint:** The later HS calculation treats class `0` as normal and classes `1–3` as abnormal. This means the final HS score measures binary detection, not four-class classification quality.

In [13]:
def get_label(crackle, wheeze):
    if crackle == 0 and wheeze == 0:
        return 0  # normal
    elif crackle == 1 and wheeze == 0:
        return 1  # crackle
    elif crackle == 0 and wheeze == 1:
        return 2  # wheeze
    else:
        return 3  # both


### Load Metadata and Extract Training and Test Embeddings

The `split_metadata.csv` is separated into the supplied training and test partitions. This cell extracts CNN14 embeddings for each training row and creates the corresponding four-class labels from the crackle and wheeze annotations.

**Hint:** Verify `AUDIO_PATH`, the CSV path, and the column names (`split`, `source_file`, `crackle`, and `wheeze`). Feature extraction may take some time; check that each audio path resolves and that the progress bar completes.

In [20]:
X, y = [], []

AUDIO_PATH = "../datasets/audios"
df = pd.read_csv("../datasets/split_metadata.csv")

train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

X_train, y_train = [], []

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Extracting Train Features"):
    path = os.path.join(AUDIO_PATH, row["source_file"]+".wav")
    
    feat = extract_panns_features(path)
    label = get_label(row["crackle"], row["wheeze"])
    
    X_train.append(feat)
    y_train.append(label)


Extracting Train Features: 100%|███████████████████████████████████████████████████| 4142/4142 [04:28<00:00, 15.44it/s]


In [21]:
X_test, y_test = [], []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Extracting Test Features"):
    path = os.path.join(AUDIO_PATH, row["source_file"]+".wav")
    
    feat = extract_panns_features(path)
    label = get_label(row["crackle"], row["wheeze"])
    
    X_test.append(feat)
    y_test.append(label)

Extracting Test Features: 100%|████████████████████████████████████████████████████| 2756/2756 [02:43<00:00, 16.84it/s]


In [22]:

X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = np.array(X_test)
y_test = np.array(y_test)

In [23]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [24]:

pca = PCA(n_components=128)   # try 64, 128, 256
X_train = pca.fit_transform(X_train)
X_test  = pca.transform(X_test)

In [25]:
print(np.unique(y_train, return_counts=True))

(array([0, 1, 2, 3]), array([2063, 1215,  501,  363]))


### Configure the SVM Classifier

The classifier uses an RBF kernel, balanced class weights, and probability estimation. The RBF kernel can model nonlinear boundaries in the PCA feature space.

**Hint:** `probability=True` enables probability estimates but can increase training time. The current evaluation uses predicted class labels, not probability thresholds.

In [26]:
clf = SVC(
    kernel="rbf",
    class_weight="balanced",
    probability=True
)

In [27]:
clf.fit(X_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [28]:
y_pred = clf.predict(X_test)

In [29]:
def compute_hs(y_true, y_pred):
    y_true_bin = (y_true != 0).astype(int)
    y_pred_bin = (y_pred != 0).astype(int)

    TP = ((y_true_bin == 1) & (y_pred_bin == 1)).sum()
    TN = ((y_true_bin == 0) & (y_pred_bin == 0)).sum()
    FP = ((y_true_bin == 0) & (y_pred_bin == 1)).sum()
    FN = ((y_true_bin == 1) & (y_pred_bin == 0)).sum()

    sensitivity = TP / (TP + FN + 1e-6)
    specificity = TN / (TN + FP + 1e-6)
    print(f"sensitivity={sensitivity}\nspecificity={specificity}\n")
    
    return (2 * sensitivity * specificity) / (sensitivity + specificity)

In [30]:
hs = compute_hs(y_test, y_pred)
print("HS Score:", hs)

sensitivity=0.5887850462287297
specificity=0.4958834702369325

HS Score: 0.538355760336588


In [31]:
# Save the fitted SVM pipeline and metadata to a .pt checkpoint

MODEL_DIR = Path("../models/cnn14+svm/")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = MODEL_DIR / "pt_cnn14_svm.pt"

checkpoint = {
    "model": clf,
    "scaler": scaler,
    "pca": pca,
    "model_type": "SVC",
    "kernel": "rbf",
    "class_weight": "balanced",
    "probability": True,
    "sample_rate": 16000,
    "max_audio_seconds": 10,
    "pca_components": 128,
    "class_names": {
        0: "Normal",
        1: "Crackle only",
        2: "Wheeze only",
        3: "Crackle and wheeze",
    },
}

torch.save(checkpoint, checkpoint_path)
print(f"Checkpoint saved to: {checkpoint_path}")

Checkpoint saved to: ..\models\cnn14+svm\pt_cnn14_svm.pt
